# Soft Robotic Arm Plant — Standalone MuJoCo Notebook

This self-contained notebook builds a fast MuJoCo plant model of the fabric
pneumatic arm. It includes the robot geometry, pressure dynamics, three
open-loop motion examples, plots, and a video. It is **not a controller**.

The arm has four radial segments—East (+x), North (+y), West (−x), and
South (−y)—with five pouches from mount to tip in each segment. Pressure is
represented by `P[segment, pouch]`, a `(4, 5)` matrix in psi. The examples
apply one segment value to all five pouches to create smooth whole-arm motion.

The arm hangs toward −z: x/y describe lateral bending and z describes axial
motion. Plots use displacement from the settled home position.



## 1. Colab and local setup

Run this cell first. It selects EGL for Linux/Colab and CGL for macOS
before MuJoCo is imported. If MuJoCo was already imported in this runtime,
restart the kernel/runtime once before choosing Run all.



In [ ]:
# Install only external Python packages; all robot code and parameters live in this notebook.
import os
import platform

if platform.system() == "Linux":
    # Colab and other headless Linux runtimes.
    os.environ["MUJOCO_GL"] = "egl"
    os.environ["PYOPENGL_PLATFORM"] = "egl"
elif platform.system() == "Darwin":
    # Native off-screen rendering on macOS.
    os.environ["MUJOCO_GL"] = "cgl"
    os.environ.pop("PYOPENGL_PLATFORM", None)
else:
    # Portable fallback, including Windows desktop sessions.
    os.environ["MUJOCO_GL"] = "glfw"
    os.environ.pop("PYOPENGL_PLATFORM", None)

print(f"MuJoCo renderer: {os.environ['MUJOCO_GL']} ({platform.system()})")

%pip install -q "mujoco>=3.5.0" "imageio[ffmpeg]>=2.37.2" matplotlib numpy



## 2. Embedded parameters

`ArmConfig` keeps all model values in one place:

| Parameter group | What it controls |
|---|---|
| Geometry and mass | Arm dimensions, spacer, distributed mass, and tip frame |
| Pressure gains | Pressure-to-bending and pressure-to-axial force conversion |
| Passive mechanics | Bending/axial stiffness and damping |
| Pneumatics and solver | Pressure limit, delay, response time, and timestep |

The spacer changes placement only. Higher stiffness reduces deformation, higher
damping suppresses oscillation, and a larger pneumatic time constant slows the
response. The values should be re-identified if the physical arm changes.



In [ ]:
# Standard-library tools for pressure-history storage and model configuration.
from collections import deque
from dataclasses import dataclass

# External packages used for video, plotting, physics, and array operations.
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import mujoco
import numpy as np


# All physical and numerical parameters are grouped here so the model can be
# modified without searching through the simulator implementation.
@dataclass
class ArmConfig:
    # Structure: four radial segments and five axial pouch levels.
    n_segments: int = 4
    n_pouches: int = 5

    # Geometry [m] and mass [kg].
    length: float = 0.296
    spacer_length: float = 0.118
    col_offset: float = 0.028
    col_radius: float = 0.018
    mass: float = 0.35
    tip_mass: float = 0.08
    tip_arm: float = 0.07
    hang_down: bool = True

    # Pressure-to-wrench mapping: pressure [psi] becomes force and moment.
    moment_arm: float = 0.028
    pressure_gain: float = 0.1273999973
    extension_gain: float = 0.0918534967

    # Passive mechanics represented by MuJoCo joint springs and dampers.
    base_stiffness: float = 0.4168449461
    base_damping: float = 0.3629226883
    axial_stiffness: float = 934.2301025
    axial_damping: float = 61.50955915

    # Pneumatic limits/dynamics and the internal physics timestep.
    p_max: float = 10.0          # psi
    pressure_delay: float = 0.5  # s
    tau_pneumatic: float = 0.6   # s
    timestep: float = 0.001      # s

    @property
    def n_channels(self) -> int:
        """Total number of pouch pressure inputs."""
        return self.n_segments * self.n_pouches

    def level_height(self) -> float:
        """Resting length assigned to one of the five body levels."""
        return self.length / self.n_pouches

    def col_azimuths(self) -> np.ndarray:
        """East, North, West, South column azimuths [rad]."""
        return np.deg2rad([0.0, 90.0, 180.0, 270.0])

    @property
    def mount_z(self) -> float:
        # The free tip starts 0.40 m above the visual floor.
        if self.hang_down:
            return self.length + self.spacer_length + 0.40
        return 0.05


cfg = ArmConfig()
print(f"Arm length: {cfg.length * 100:.1f} cm")
print(f"Levels: {cfg.n_pouches}; level height: {cfg.level_height() * 100:.2f} cm")
print("Column azimuths:", np.degrees(cfg.col_azimuths()).astype(int).tolist())
print("Independent pressure channels:", cfg.n_channels)



## 3. MJCF model

The fabric arm is approximated by five nested rigid-body levels. This is much
faster than a finite-element model and is suitable for dynamic simulation and
controller development, but it does not reproduce the detailed fabric shape.

Each level contains three compliant coordinates:

- `ext[k]`: axial slide;
- `bx[k]`, `by[k]`: bending hinges.

This gives **15 mechanical DOF**. Springs and dampers represent passive
compliance; MuJoCo handles gravity and inertia. The colored capsules show segment
locations but are not deformable pouches. The nested chain makes base motion affect
all downstream levels, and the `tip` site provides pose and velocity sensors.



In [ ]:
def build_arm_xml(cfg: ArmConfig) -> str:
    """Build the complete arm MJCF as a string.

    Nested bodies make motion at one level affect every level below it.
    """
    # Divide the measured arm length and mass across five model levels.
    h = cfg.level_height()
    n_levels = cfg.n_pouches
    link_mass = cfg.mass / n_levels
    # Negative z places the arm below the mounting plate.
    zdir = -1.0 if cfg.hang_down else 1.0
    col_phis = cfg.col_azimuths()
    # Segment colors are reused at every level for visual orientation.
    col_colors = [
        "0.92 0.55 0.08 1",  # Segment 1 / East
        "0.10 0.68 0.18 1",  # Segment 2 / North
        "0.82 0.18 0.08 1",  # Segment 3 / West
        "0.08 0.28 0.90 1",  # Segment 4 / South
    ]
    disc_r = cfg.col_offset + cfg.col_radius + 0.003

    # Generate one nested body for each pouch level.
    body_xml = ""
    indent = "      "
    for k in range(n_levels):
        # Level 0 begins below the 118 mm visual spacer.
        offset = zdir * cfg.spacer_length if k == 0 else zdir * h
        body_xml += f'{indent}<body name="level{k}" pos="0 0 {offset:.6f}">\n'
        # One slide gives axial compliance at this level.
        body_xml += (
            f'{indent}  <joint name="ext{k}" type="slide" axis="0 0 {zdir:g}" '
            f'stiffness="{cfg.axial_stiffness}" damping="{cfg.axial_damping}" '
            'springref="0" range="-0.005 0.030"/>\n'
        )
        # Two perpendicular hinges allow bending in any horizontal direction.
        body_xml += (
            f'{indent}  <joint name="bx{k}" type="hinge" axis="1 0 0" '
            f'damping="{cfg.base_damping}" stiffness="{cfg.base_stiffness}" springref="0"/>\n'
            f'{indent}  <joint name="by{k}" type="hinge" axis="0 1 0" '
            f'damping="{cfg.base_damping}" stiffness="{cfg.base_stiffness}" springref="0"/>\n'
        )
        body_xml += (
            f'{indent}  <geom name="ring{k}" type="cylinder" pos="0 0 0" '
            f'size="{disc_r:.4f} 0.0025" mass="0.006" '
            'rgba="0.10 0.10 0.12 1" contype="0" conaffinity="0"/>\n'
        )
        # Colored capsules show segment locations and carry distributed mass.
        col_mass = link_mass * 0.85 / cfg.n_segments
        for s, phi in enumerate(col_phis):
            cx = cfg.col_offset * np.cos(phi)
            cy = cfg.col_offset * np.sin(phi)
            body_xml += (
                f'{indent}  <geom name="col{s}_l{k}" type="capsule" '
                f'fromto="{cx:.5f} {cy:.5f} 0 {cx:.5f} {cy:.5f} {zdir * h:.5f}" '
                f'size="{cfg.col_radius:.4f}" mass="{col_mass:.6f}" '
                f'rgba="{col_colors[s]}" contype="0" conaffinity="0"/>\n'
            )
        # The central core carries the remaining mass at this level.
        body_xml += (
            f'{indent}  <geom name="core{k}" type="cylinder" '
            f'pos="0 0 {zdir * h * 0.5:.5f}" '
            f'size="0.005 {h * 0.48:.5f}" mass="{link_mass * 0.15:.6f}" '
            'rgba="0.12 0.12 0.14 1" contype="0" conaffinity="0"/>\n'
        )
        indent += "  "

    # Approximate the connector disc and OptiTrack marker frame at the tip.
    a = cfg.tip_arm
    body_xml += (
        f'{indent}<body name="tip_disc" pos="0 0 {zdir * h:.6f}">\n'
        f'{indent}  <geom name="tip_ring" type="cylinder" pos="0 0 0" '
        f'size="{disc_r:.4f} 0.0025" mass="0.010" '
        'rgba="0.10 0.10 0.12 1" contype="0" conaffinity="0"/>\n'
        f'{indent}  <body name="tip_frame" pos="0 0 {zdir * 0.012:.6f}">\n'
        f'{indent}    <geom name="tip_bar_x" type="capsule" '
        f'fromto="-{a} 0 0 {a} 0 0" size="0.004" '
        f'mass="{cfg.tip_mass * 0.30:.4f}" rgba="0.12 0.12 0.14 1" '
        'contype="0" conaffinity="0"/>\n'
        f'{indent}    <geom name="tip_bar_y" type="capsule" '
        f'fromto="0 -{a} 0 0 {a} 0" size="0.004" '
        f'mass="{cfg.tip_mass * 0.30:.4f}" rgba="0.12 0.12 0.14 1" '
        'contype="0" conaffinity="0"/>\n'
        f'{indent}    <geom name="tip_ball_c" type="sphere" pos="0 0 0" '
        f'size="0.007" mass="{cfg.tip_mass * 0.40:.4f}" '
        'rgba="0.85 0.85 0.85 1" contype="0" conaffinity="0"/>\n'
        f'{indent}    <geom name="tip_ball_px" type="sphere" pos="{a} 0 0" '
        'size="0.005" mass="0.001" rgba="0.85 0.85 0.85 1" contype="0" conaffinity="0"/>\n'
        f'{indent}    <geom name="tip_ball_nx" type="sphere" pos="-{a} 0 0" '
        'size="0.005" mass="0.001" rgba="0.85 0.85 0.85 1" contype="0" conaffinity="0"/>\n'
        f'{indent}    <geom name="tip_ball_py" type="sphere" pos="0 {a} 0" '
        'size="0.005" mass="0.001" rgba="0.85 0.85 0.85 1" contype="0" conaffinity="0"/>\n'
        f'{indent}    <geom name="tip_ball_ny" type="sphere" pos="0 -{a} 0" '
        'size="0.005" mass="0.001" rgba="0.85 0.85 0.85 1" contype="0" conaffinity="0"/>\n'
        f'{indent}    <site name="tip" pos="0 0 0" size="0.007" rgba="0.1 0.9 0.3 1"/>\n'
        f'{indent}  </body>\n'
        f'{indent}</body>\n'
    )
    # Close the five nested level bodies after attaching the tip.
    for k in range(n_levels, 0, -1):
        body_xml += "      " + "  " * (k - 1) + "</body>\n"

    # Add the fixed mount, floor, lighting, and sensors around the generated chain.
    return f"""<mujoco model="fabric_soft_arm">
  <option timestep="{cfg.timestep}" gravity="0 0 -9.81" integrator="implicitfast"/>
  <visual>
    <headlight ambient="0.45 0.45 0.48" diffuse="0.7 0.7 0.7" specular="0.1 0.1 0.1"/>
    <global offwidth="1280" offheight="720"/>
    <quality shadowsize="2048"/>
  </visual>
  <asset>
    <texture name="grid" type="2d" builtin="checker" width="512" height="512"
             rgb1="0.18 0.20 0.24" rgb2="0.24 0.26 0.30"/>
    <material name="grid" texture="grid" texrepeat="8 8" reflectance="0.08"/>
    <material name="mount_wood" rgba="0.62 0.48 0.30 1"/>
    <material name="post_metal" rgba="0.14 0.14 0.16 1"/>
  </asset>
  <worldbody>
    <light pos="0.0 -0.8 1.8" dir="0.0 0.5 -1" diffuse="0.6 0.6 0.65"/>
    <light pos="0.6 0.4 1.5" dir="-0.4 -0.3 -1" diffuse="0.4 0.4 0.45"/>
    <geom name="floor" type="plane" size="2.0 2.0 0.05" material="grid"/>
    <body name="mount" pos="0 0 {cfg.mount_z:.4f}">
      <geom name="mount_plate" type="box" size="0.150 0.150 0.009"
            material="mount_wood" contype="0" conaffinity="0"/>
      <geom name="spacer_fl" type="cylinder" fromto="-0.035 -0.035 0 -0.035 -0.035 -{cfg.spacer_length:.4f}"
            size="0.006" material="post_metal" contype="0" conaffinity="0"/>
      <geom name="spacer_fr" type="cylinder" fromto="0.035 -0.035 0 0.035 -0.035 -{cfg.spacer_length:.4f}"
            size="0.006" material="post_metal" contype="0" conaffinity="0"/>
      <geom name="spacer_bl" type="cylinder" fromto="-0.035 0.035 0 -0.035 0.035 -{cfg.spacer_length:.4f}"
            size="0.006" material="post_metal" contype="0" conaffinity="0"/>
      <geom name="spacer_br" type="cylinder" fromto="0.035 0.035 0 0.035 0.035 -{cfg.spacer_length:.4f}"
            size="0.006" material="post_metal" contype="0" conaffinity="0"/>
      <geom name="mount_disc" type="cylinder" pos="0 0 -{cfg.spacer_length:.4f}"
            size="{disc_r:.4f} 0.003" mass="0.010"
            rgba="0.10 0.10 0.12 1" contype="0" conaffinity="0"/>
{body_xml}    </body>
  </worldbody>
  <sensor>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
    <framequat name="tip_quat" objtype="site" objname="tip"/>
    <framelinvel name="tip_vel" objtype="site" objname="tip"/>
  </sensor>
</mujoco>"""


# Compile once as an early MJCF check.
_model_check = mujoco.MjModel.from_xml_string(build_arm_xml(cfg))
print(f"MJCF loaded: {_model_check.nq} DOF, {_model_check.nbody} bodies, {_model_check.ngeom} geoms")



## 4. Plant simulator

`SoftArmSim` maps the `(4, 5)` pressure field to the 15 joint coordinates.
At 100 Hz, each call to `step()` performs ten internal 1 ms physics steps.
Pressure is delayed by 0.5 s and then follows a 0.6 s first-order response:

`P_actual ← P_actual + alpha (P_delayed − P_actual)`

Here `alpha` is set by the 1 ms timestep and the pneumatic time constant.

At level `k`, segment `s` has bending direction
`a[s] = (−sin(phi[s]), cos(phi[s]))`. The applied loads are

`M[k] = pressure_gain × moment_arm × sum_s(P_actual[s,k] × a[s])`

`F_axial[k] = extension_gain × sum_s(P_actual[s,k])`

Equal segment pressures cancel bending and add axial force; unequal pressures
produce bending. This calculation is applied at all five pouch levels.

`SoftArmSim` only advances the plant. It contains no target trajectory,
tracking error, feedback calculation, or controller gains.



In [ ]:
class SoftArmSim:
    """MuJoCo plant with one independently commanded pressure per pouch.

    command_psi has shape (4, 5). Rows are Segments 1 through 4
    (East, North, West, South), and columns are Pouches 1 through 5
    from the mount toward the free tip.
    """

    def __init__(self, step_hz: float = 100.0):
        if not np.isfinite(step_hz) or step_hz <= 0:
            raise ValueError("step_hz must be positive")

        self.cfg = ArmConfig()
        # The public plant runs at 100 Hz while MuJoCo integrates at 1 kHz.
        self.step_dt = 1.0 / float(step_hz)
        self.n_sub_steps = max(1, round(self.step_dt / self.cfg.timestep))
        self.pressure_delay_steps = round(
            self.cfg.pressure_delay / self.step_dt
        )
        # This queue implements the measured command-to-pressure transport delay.
        self._pressure_delay_queue: deque[np.ndarray] = deque()

        self.model = mujoco.MjModel.from_xml_string(build_arm_xml(self.cfg))
        self.data = mujoco.MjData(self.model)

        # Convert joint names in the generated MJCF into force-vector indices.
        def dof(name: str) -> int:
            joint_id = mujoco.mj_name2id(
                self.model, mujoco.mjtObj.mjOBJ_JOINT, name
            )
            return int(self.model.jnt_dofadr[joint_id])

        self.ext_dofs = np.array(
            [dof(f"ext{k}") for k in range(self.cfg.n_pouches)]
        )
        self.bend_dofs = np.array(
            [(dof(f"bx{k}"), dof(f"by{k}")) for k in range(self.cfg.n_pouches)]
        )

        # Each segment pressure produces a moment tangent to its radial position.
        phis = self.cfg.col_azimuths()
        self.col_axes = np.stack([-np.sin(phis), np.cos(phis)], axis=1)
        if self.cfg.hang_down:
            self.col_axes = -self.col_axes

        shape = (self.cfg.n_segments, self.cfg.n_pouches)
        self.command_psi = np.zeros(shape)
        self.p_actual = np.zeros(shape)
        self.reset()

    def _reset_delay_queue(self) -> None:
        """Fill the delay line with zero-pressure history after reset."""
        self._pressure_delay_queue.clear()
        zero = np.zeros_like(self.p_actual)
        for _ in range(self.pressure_delay_steps):
            self._pressure_delay_queue.append(zero.copy())

    def _validate_command(self, command_psi) -> np.ndarray:
        """Require one finite command per pouch and enforce pressure limits."""
        command = np.asarray(command_psi, dtype=float)
        expected = (self.cfg.n_segments, self.cfg.n_pouches)
        if command.shape != expected:
            raise ValueError(
                f"command_psi must have shape {expected}; got {command.shape}"
            )
        if not np.all(np.isfinite(command)):
            raise ValueError("command_psi must contain only finite values")
        return np.clip(command, 0.0, self.cfg.p_max)

    def reset(self) -> dict:
        """Restore mechanical, pneumatic, and delay states to rest."""
        mujoco.mj_resetData(self.model, self.data)
        self.command_psi.fill(0.0)
        self.p_actual.fill(0.0)
        self._reset_delay_queue()
        mujoco.mj_forward(self.model, self.data)
        return self.observe()

    def step(self, command_psi) -> dict:
        """Advance the plant by one sample using a (4, 5) pressure matrix."""
        self.command_psi[:] = self._validate_command(command_psi)
        # Push the newest command in and apply the command leaving the delay line.
        self._pressure_delay_queue.append(self.command_psi.copy())
        delayed_target = self._pressure_delay_queue.popleft()

        # Forward-Euler update of the first-order pouch-pressure state.
        alpha = self.cfg.timestep / max(
            self.cfg.tau_pneumatic, self.cfg.timestep
        )
        for _ in range(self.n_sub_steps):
            self.p_actual += alpha * (delayed_target - self.p_actual)
            self._apply_pressure_wrench()
            mujoco.mj_step(self.model, self.data)

        return self.observe()

    def observe(self) -> dict:
        """Return copies of signals available to a future controller."""
        sensor = self.data.sensordata
        return {
            "time": float(self.data.time),
            "tip_pos": sensor[0:3].copy(),
            "tip_quat": sensor[3:7].copy(),
            "tip_vel": sensor[7:10].copy(),
            "pouch_pressures": self.p_actual.copy(),
            "p_actual": self.p_actual.copy(),
            "q": self.data.qpos.copy(),
        }

    def _apply_pressure_wrench(self) -> None:
        """Convert all pouch pressures into joint forces for one substep."""
        qf = self.data.qfrc_applied
        qf[:] = 0.0
        gain = self.cfg.pressure_gain * self.cfg.moment_arm
        # Matrix multiplication sums the four segment moments at every level.
        moments = gain * (self.p_actual.T @ self.col_axes)
        for level in range(self.cfg.n_pouches):
            bx, by = self.bend_dofs[level]
            qf[bx] += moments[level, 0]
            qf[by] += moments[level, 1]
            # All four pressures at a level contribute to axial loading.
            qf[self.ext_dofs[level]] += (
                self.cfg.extension_gain * self.p_actual[:, level].sum()
            )


# Executable interface and independence checks.
sim = SoftArmSim()
obs = sim.reset()
assert sim.model.nq == 15
assert sim.cfg.n_channels == 20
assert obs["pouch_pressures"].shape == (4, 5)
assert sim.p_actual.shape == (4, 5)
assert sim.pressure_delay_steps == 50
assert np.allclose(
    np.degrees(sim.cfg.col_azimuths()),
    [0.0, 90.0, 180.0, 270.0],
)

try:
    sim.step(np.zeros(20))
    raise AssertionError("A flat command should not be accepted")
except ValueError:
    pass

# Interface isolation check only; the motion examples use complete rows.
single_channel = np.zeros((4, 5))
single_channel[2, 3] = 5.0
for _ in range(sim.pressure_delay_steps):
    sim.step(single_channel)
assert np.allclose(sim.p_actual, 0.0)
sim.step(single_channel)
assert np.argwhere(sim.p_actual > 0.0).tolist() == [[2, 3]]

# A segment command reaches all five pouch levels in that segment.
sim.reset()
full_segment = np.zeros((4, 5))
full_segment[2, :] = 5.0
for _ in range(sim.pressure_delay_steps):
    sim.step(full_segment)
sim.step(full_segment)
assert np.argwhere(sim.p_actual > 0.0).tolist() == [
    [2, 0], [2, 1], [2, 2], [2, 3], [2, 4]
]

print("Simulator ready")
print("  pressure channels:", sim.cfg.n_channels)
print("  command shape:", sim.command_psi.shape)
print("  pressure delay:", sim.cfg.pressure_delay, "s")


## 5. Open-loop motion examples

These 0.1 Hz waveforms are predetermined inputs, not controllers. Each returns
a complete `(4, 5)` command and uses the same value across a segment's five
pouches:

- **Axial:** all four segments receive the same sinusoid, so bending cancels.
- **Circular:** a cosine pressure pattern rotates around the four segments.
- **Triangular:** three directional pressure patterns are linearly interpolated.

Every run first settles at 3.5 psi and reports home-relative motion. The axial
plot uses z displacement because its x/y motion should be nearly zero.



In [ ]:
# Shared settings for all three open-loop examples.
STEP_HZ = 100.0
MOTION_FREQUENCY_HZ = 0.1
PRESSURE_OFFSET_PSI = 3.5
PRESSURE_AMPLITUDE_PSI = 2.0
SEGMENT_LABELS = (
    "Segment 1 East",
    "Segment 2 North",
    "Segment 3 West",
    "Segment 4 South",
)
SEGMENT_COLORS = ("tab:orange", "tab:green", "tab:red", "tab:blue")


def repeat_across_pouches(segment_pressures) -> np.ndarray:
    """Apply each segment pressure to all five pouches in that segment."""
    segment_pressures = np.asarray(segment_pressures, dtype=float)
    if segment_pressures.shape != (cfg.n_segments,):
        raise ValueError("segment_pressures must contain four values")
    return np.repeat(segment_pressures[:, None], cfg.n_pouches, axis=1)


def axial_command(t: float) -> np.ndarray:
    """Inflate every segment equally so bending moments cancel."""
    pressure = PRESSURE_OFFSET_PSI + PRESSURE_AMPLITUDE_PSI * np.sin(
        2.0 * np.pi * MOTION_FREQUENCY_HZ * t
    )
    return np.full((cfg.n_segments, cfg.n_pouches), pressure)


def directional_command(angle_rad: float) -> np.ndarray:
    """Create a four-segment pressure imbalance in one XY direction."""
    # The cosine weights each segment by its alignment with the desired direction.
    by_segment = PRESSURE_OFFSET_PSI + PRESSURE_AMPLITUDE_PSI * np.cos(
        angle_rad - cfg.col_azimuths()
    )
    return repeat_across_pouches(by_segment)


def circular_command(t: float) -> np.ndarray:
    """Rotate the directional pressure pattern continuously with time."""
    angle = 2.0 * np.pi * MOTION_FREQUENCY_HZ * t
    return directional_command(angle)


def triangular_command(t: float) -> np.ndarray:
    """Interpolate between three directional patterns over one period."""
    period_s = 1.0 / MOTION_FREQUENCY_HZ
    edge = 3.0 * (t % period_s) / period_s
    start = int(edge) % 3
    fraction = edge - int(edge)
    vertex_angles = np.array([0.0, 2.0 * np.pi / 3.0, 4.0 * np.pi / 3.0])
    # Each vertex is a complete pressure matrix for one bend direction.
    vertices = np.stack(
        [directional_command(angle) for angle in vertex_angles]
    )
    return (
        (1.0 - fraction) * vertices[start]
        + fraction * vertices[(start + 1) % 3]
    )


OPEN_LOOP_MOTIONS = {
    "axial": axial_command,
    "circular": circular_command,
    "triangular": triangular_command,
}


def run_open_loop_example(
    name: str,
    duration_s: float = 1.0 / MOTION_FREQUENCY_HZ,
):
    """Apply one fixed pressure schedule and return synchronized arrays."""
    if name not in OPEN_LOOP_MOTIONS:
        raise ValueError(f"name must be one of {tuple(OPEN_LOOP_MOTIONS)}")

    example_sim = SoftArmSim(step_hz=STEP_HZ)
    # Establish a repeatable home pose before recording the trajectory.
    baseline = np.full(
        (cfg.n_segments, cfg.n_pouches),
        PRESSURE_OFFSET_PSI,
    )
    for _ in range(round(4.0 * STEP_HZ)):
        example_sim.step(baseline)
    home = example_sim.observe()["tip_pos"].copy()

    steps = round(duration_s * STEP_HZ)
    time = np.arange(steps) / STEP_HZ
    tip = np.zeros((steps, 3))
    command = np.zeros((steps, cfg.n_segments, cfg.n_pouches))
    pressure = np.zeros_like(command)
    command_fn = OPEN_LOOP_MOTIONS[name]

    # Store command, measured tip displacement, and actual pouch pressure together.
    for tick, t in enumerate(time):
        command[tick] = command_fn(float(t))
        observation = example_sim.step(command[tick])
        tip[tick] = observation["tip_pos"] - home
        pressure[tick] = observation["pouch_pressures"]

    return {
        "name": name,
        "sim": example_sim,
        "time": time,
        "tip": tip,
        "command": command,
        "pressure": pressure,
    }


def plot_open_loop_example(result) -> None:
    """Plot spatial response, time response, and pressure dynamics."""
    time = result["time"]
    tip_mm = result["tip"] * 1000.0
    command_mean = result["command"].mean(axis=2)
    pressure_mean = result["pressure"].mean(axis=2)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.7))
    if result["name"] == "axial":
        # Axial motion is along z. Its x/y values are only numerical noise.
        mean_pressure = pressure_mean.mean(axis=1)
        axes[0].plot(mean_pressure, tip_mm[:, 2], lw=1.5)
        axes[0].set(
            xlabel="mean pouch pressure [psi]",
            ylabel="tip z change [mm]",
            title="Axial pressure–displacement response",
        )
    else:
        axes[0].plot(tip_mm[:, 0], tip_mm[:, 1], lw=1.5)
        axes[0].set(
            xlabel="tip x change [mm]",
            ylabel="tip y change [mm]",
            title="Home-relative tip trajectory",
        )
        axes[0].axis("equal")
    axes[0].grid(alpha=0.3)

    for index, label in enumerate(("x", "y", "z")):
        axes[1].plot(time, tip_mm[:, index], label=label)
    axes[1].set(
        xlabel="time [s]",
        ylabel="tip change [mm]",
        title="Tip response",
    )
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    for index, (label, color) in enumerate(
        zip(SEGMENT_LABELS, SEGMENT_COLORS)
    ):
        axes[2].plot(
            time,
            command_mean[:, index],
            color=color,
            linestyle="--",
            alpha=0.75,
            label=f"{label} command",
        )
        axes[2].plot(
            time,
            pressure_mean[:, index],
            color=color,
            linewidth=1.2,
            label=f"{label} pressure",
        )
    axes[2].set(
        xlabel="time [s]",
        ylabel="pressure [psi]",
        title="Mean pressure across each segment's five pouches",
    )
    axes[2].legend(fontsize=7, ncol=2)
    axes[2].grid(alpha=0.3)

    fig.suptitle(f"{result['name'].capitalize()} open-loop motion")
    fig.tight_layout()
    plt.show()


# Validate the three command functions before running the examples.
for command_fn in OPEN_LOOP_MOTIONS.values():
    for check_time in (0.0, 2.5, 7.5):
        check = command_fn(check_time)
        assert check.shape == (4, 5)
        assert np.all(np.isfinite(check))
        assert np.all((0.0 <= check) & (check <= cfg.p_max))
        # Motion examples always actuate complete segment rows.
        assert np.allclose(check, check[:, :1])

# Run all three examples. Comment out names that are not needed.
results = {}
for motion_name in ("axial", "circular", "triangular"):
    print("Running", motion_name)
    results[motion_name] = run_open_loop_example(motion_name)
    plot_open_loop_example(results[motion_name])


## 6. Simulation video

This cell renders one complete 10-second circular cycle at 0.1 Hz. The
pressure waveform is exactly the same predetermined open-loop example used
above; the video adds visualization only and does not add a controller.
Frames are streamed directly into the MP4 to keep Colab memory use low.



In [ ]:
from pathlib import Path
from IPython.display import Video, display

# Video settings are independent of the 100 Hz physics/control rate.
VIDEO_PATH = Path("soft_arm_circular_open_loop.mp4")
VIDEO_FPS = 20
VIDEO_WIDTH = 640
VIDEO_HEIGHT = 480
VIDEO_DURATION_S = 1.0 / MOTION_FREQUENCY_HZ

video_sim = SoftArmSim(step_hz=STEP_HZ)
# Settle at the same baseline used by the plotted examples.
baseline = np.full(
    (cfg.n_segments, cfg.n_pouches),
    PRESSURE_OFFSET_PSI,
)
for _ in range(round(4.0 * STEP_HZ)):
    video_sim.step(baseline)

# Create an off-screen renderer so the cell also works in Colab.
renderer = mujoco.Renderer(
    video_sim.model,
    height=VIDEO_HEIGHT,
    width=VIDEO_WIDTH,
)
# Aim a fixed diagonal camera at the middle of the hanging arm.
camera = mujoco.MjvCamera()
mujoco.mjv_defaultCamera(camera)
camera.azimuth = 135.0
camera.elevation = -18.0
camera.distance = 0.50
arm_base_z = video_sim.cfg.mount_z - video_sim.cfg.spacer_length
camera.lookat[:] = [0.0, 0.0, arm_base_z - video_sim.cfg.length * 0.5]

assert STEP_HZ % VIDEO_FPS == 0
# Simulate every 100 Hz step but save only the requested video frames.
frame_stride = max(1, round(STEP_HZ / VIDEO_FPS))
video_steps = round(VIDEO_DURATION_S * STEP_HZ)
# Stream frames directly to MP4 instead of keeping the full video in memory.
try:
    with imageio.get_writer(
        VIDEO_PATH,
        fps=VIDEO_FPS,
        codec="libx264",
        pixelformat="yuv420p",
    ) as writer:
        for tick in range(video_steps):
            t = tick / STEP_HZ
            video_sim.step(circular_command(t))
            if tick % frame_stride == 0:
                renderer.update_scene(video_sim.data, camera=camera)
                writer.append_data(renderer.render())
finally:
    renderer.close()

print("Saved", VIDEO_PATH)
display(
    Video(
        filename=str(VIDEO_PATH),
        embed=True,
        html_attributes="controls loop",
    )
)



## 7. Plant interface

The pressure command always has shape `(4, 5)`:

- Rows 0–3: Segment 1 East, Segment 2 North, Segment 3 West, Segment 4 South.
- Columns 0–4: Pouch 1 at the mount through Pouch 5 at the free tip.

Use `command_psi[segment, :]` to actuate every pouch level in a segment.
The motion examples above generate their tip paths using this full-row command.

```python
sim = SoftArmSim()
command_psi = np.full((4, 5), 3.5)
command_psi[1, :] = 5.0  # all five pouches in Segment 2
observation = sim.step(command_psi)

tip_position = observation["tip_pos"]
pouch_pressures = observation["pouch_pressures"]
```

`observation` also contains tip orientation, tip velocity, joint state, and
simulation time. Pressure commands are limited to 0–10 psi. A separate
controller can use this interface without modifying the plant model.



## 8. Model scope

This model includes gravity, inertia, compliant joints, pressure lag, all 20
pouch inputs, and tip sensing. It intentionally omits detailed fabric
deformation, airflow and pouch-volume thermodynamics, hysteresis, leakage, and
manufacturing asymmetry.

Use it as a fast controller-development plant and validate any controller on
the physical robot before deployment. A controller can read `observation`,
compute a new `(4, 5)` command, and call `step()` without changing this model.

